# Smart Agriculture Data Analytics using Apache Spark

This notebook analyses the Government of India district-wise crop production dataset (1997–2015), containing approximately 246,091 crop records across states, districts, seasons and crops. It performs Spark-based cleaning, aggregation, trend analysis and production regression.

**Source:** Government of India, Ministry of Agriculture & Farmers Welfare, via data.gov.in. A public GitHub mirror is used for a Colab-friendly download because the government portal may block automated downloads.

**Execution:** Google Colab local Spark mode (not a multi-node cluster).

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q install pyspark==3.5.1

import os, subprocess
java_home = subprocess.check_output('dirname $(dirname $(readlink -f $(which java)))', shell=True, text=True).strip()
os.environ['JAVA_HOME'] = java_home
print('JAVA_HOME:', java_home)
print('PySpark is ready')

In [ ]:
!git clone https://github.com/deecares/Smart-Agriculture-Crop-Recommendation.git
%cd Smart-Agriculture-Crop-Recommendation
!mkdir -p data

## Download the crop production dataset

The notebook downloads the CSV from a public GitHub mirror. If that mirror is unavailable, a file-upload fallback is provided.

In [ ]:
import os, requests
from google.colab import files
target_path = 'data/crop_production.csv'
download_urls = [
  'https://raw.githubusercontent.com/JitendraAmbekar/india-crop-production-analysis/main/crop_production.csv',
  'https://raw.githubusercontent.com/Aliabdurahman/Prediction-of-crop-Production-in-India/master/crop_production.csv'
]
downloaded = False
for url in download_urls:
    try:
        print('Trying:', url)
        response = requests.get(url, headers={'User-Agent':'Mozilla/5.0'}, timeout=180, stream=True)
        response.raise_for_status()
        total = 0
        with open(target_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk); total += len(chunk)
        if total > 1_000_000:
            downloaded = True
            print('Dataset downloaded:', round(total/(1024**2),2), 'MiB')
            break
        os.remove(target_path)
    except Exception as e:
        if os.path.exists(target_path): os.remove(target_path)
        print('Download attempt failed:', e)
if not downloaded:
    print('Automatic download failed. Upload crop_production.csv now.')
    uploaded = files.upload()
    if not uploaded: raise RuntimeError('No dataset uploaded.')
    name = next(iter(uploaded)); os.replace(name, target_path)
print('Saved dataset:', target_path, 'Size (MiB):', round(os.path.getsize(target_path)/(1024**2),2))
if os.path.getsize(target_path) < 1_000_000: raise ValueError('File is too small; expected the full crop production CSV.')

In [ ]:
import pandas as pd
preview = pd.read_csv('data/crop_production.csv', nrows=5)
print('Columns:', list(preview.columns))
display(preview)

In [ ]:
!spark-submit src/agriculture_analytics.py --input data/crop_production.csv --output outputs

In [ ]:
import glob
metric_files = glob.glob('outputs/model_metrics/part-*.csv')
if not metric_files: raise FileNotFoundError('Model metrics were not created; inspect the Spark output above.')
metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
display(metrics.sort_values('rmse'))
print('Output folders:', sorted(os.listdir('outputs')))